# Getting Started with Distributed LLM Fine-tuning

This notebook provides an interactive introduction to the Distributed LLM Fine-tuning Platform.

**What you'll learn:**
1. Loading and configuring models
2. Preparing datasets
3. Running training
4. Evaluating results
5. Visualizing training progress

**Prerequisites:**
- Python 3.8+
- GPU recommended (but works on CPU)
- ~16GB RAM

Let's get started! 🚀

## 1. Setup and Imports

In [ ]:
# Install requirements if needed
# !pip install -r ../requirements.txt

import sys
sys.path.append('..')

from src.training.config import TrainingConfig
from src.training.trainer import Trainer
from src.evaluation.metrics import LLMEvaluator
from src.utils.visualization import TrainingVisualizer
from src.utils.dataset_analysis import quick_analysis

print("✓ Imports successful!")

## 2. Analyze Dataset

Before training, let's understand our dataset.

In [ ]:
# Analyze WikiText dataset
stats = quick_analysis("wikitext", "wikitext-2-raw-v1")

# Key insights:
print(f"\nDataset has {stats['total_samples']:,} samples")
print(f"Average length: {stats['text_statistics']['avg_word_length']:.1f} words")
print(f"Vocabulary: {stats['text_statistics']['vocabulary_size']:,} unique words")

## 3. Configure Training

Create a training configuration with sensible defaults.

In [ ]:
config = TrainingConfig(
    # Model
    model_name="gpt2",  # Small model for quick training
    
    # Dataset
    dataset_name="wikitext",
    dataset_config="wikitext-2-raw-v1",
    max_seq_length=256,  # Shorter for speed
    
    # Training
    num_epochs=1,  # Just 1 epoch for demo
    batch_size=4,  # Small batch for memory
    learning_rate=5e-5,
    
    # Optimization
    precision="fp16",  # Mixed precision for speed
    
    # Output
    output_dir="../outputs/notebook_demo",
    logging_steps=5,
    
    # MLflow (optional)
    use_mlflow=False,  # Disable for notebook
)

# Display configuration
print("Training Configuration:")
print(f"  Model: {config.model_name}")
print(f"  Dataset: {config.dataset_name}")
print(f"  Epochs: {config.num_epochs}")
print(f"  Batch Size: {config.batch_size}")
print(f"  Learning Rate: {config.learning_rate}")

# Estimate memory usage
memory = config.estimate_memory_usage(124_000_000)  # GPT-2 has 124M params
print(f"\nEstimated GPU Memory: {memory['total_gb']:.2f} GB")

## 4. Initialize Trainer

In [ ]:
# Create trainer
trainer = Trainer(config)

print("✓ Trainer initialized")
print(f"  Device: {trainer.device}")

## 5. Load Model and Data

In [ ]:
# Load model
print("Loading model...")
trainer.load_model_and_tokenizer()

# Load and prepare data
print("\nPreparing datasets...")
trainer.prepare_datasets()

# Setup optimizer
print("\nSetting up optimizer...")
trainer.setup_optimizer_and_scheduler()

print("\n✓ Ready to train!")

## 6. Run Training

Note: This may take 5-15 minutes depending on your hardware.

In [ ]:
# Run training
results = trainer.train()

print("\n" + "=" * 80)
print("Training Complete!")
print("=" * 80)
print(f"Final Loss: {results['final_loss']:.4f}")
print(f"Best Eval Loss: {results['best_eval_loss']:.4f}")
print(f"Total Steps: {results['total_steps']}")
print(f"Training Time: {results['training_time'] / 60:.2f} minutes")

## 7. Evaluate Model

In [ ]:
# Run evaluation
eval_results = trainer.evaluate()

print("Evaluation Results:")
print(f"  Loss: {eval_results['loss']:.4f}")
print(f"  Perplexity: {eval_results['perplexity']:.2f}")

## 8. Visualize Training (Optional)

Visualize training progress if you saved metrics.

In [ ]:
# Example: Create training history visualization
# (In practice, you'd load this from saved training logs)

visualizer = TrainingVisualizer()

# Example history (replace with actual data)
example_history = {
    "train_loss": [2.5, 2.3, 2.1, 1.9, 1.8, 1.7],
    "val_loss": [2.6, 2.4, 2.2, 2.0, 1.9, 1.85],
    "learning_rate": [5e-5, 4.8e-5, 4.5e-5, 4.2e-5, 3.8e-5, 3.5e-5],
}

fig = visualizer.plot_training_history(example_history)
# fig.show()  # Uncomment to display

## 9. Generate Text (Optional)

Test the fine-tuned model by generating text.

In [ ]:
# Generate text
prompt = "The future of artificial intelligence is"
inputs = trainer.tokenizer(prompt, return_tensors="pt").to(trainer.device)

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_length=100,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=2,
        pad_token_id=trainer.tokenizer.eos_token_id
    )

generated_text = trainer.tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\nGenerated Text:")
print(generated_text)

## Next Steps

Congratulations! You've completed your first fine-tuning run. 🎉

**What to try next:**

1. **Experiment with hyperparameters**:
   - Increase `num_epochs` for better quality
   - Try different `learning_rate` values
   - Use `use_lora=True` for efficient fine-tuning

2. **Try different models**:
   - "gpt2-medium" (355M params)
   - "gpt2-large" (774M params)
   - "EleutherAI/gpt-neo-125M"

3. **Use your own dataset**:
   - Load custom data with `load_dataset()`
   - Prepare text for your use case

4. **Run hyperparameter optimization**:
   - See `02_hyperparameter_tuning.ipynb`

5. **Deploy your model**:
   - See `03_model_deployment.ipynb`

**Documentation**:
- [Learning Guide](../docs/guides/LEARNING_GUIDE.md)
- [Architecture](../docs/ARCHITECTURE.md)
- [FAQ](../docs/guides/FAQ.md)